# SentenceKDTransformer — ablations and document-level inference

Companion notebook for `examples/medical_transcriptions_classification_sentence_kd_transformer.py`.

**Paper:** Kim et al., *Integrating ChatGPT into Secure Hospital Networks: A Case Study on Improving Radiology Report Analysis*, CHIL 2024 — [proceedings.mlr.press/v248/kim24a](https://proceedings.mlr.press/v248/kim24a.html).

**What this notebook shows**
1. Instantiates the PyHealth `SentenceKDTransformer` model.
2. Runs a λ (contrastive-weight) sweep on an existing PyHealth dataset — a novel ablation beyond the paper's λ∈{0, 1} test.
3. Runs a τ (contrastive temperature) sweep — never varied in the paper.
4. Demonstrates the document-level max-pool inference from paper Eq. 4 on a simulated radiology report.

**Backbone:** `StanfordAIMI/RadBERT` — the paper's primary clinical BERT (~110M parameters, ~440 MB download on first run).

**Runtime:** ~3–5 minutes on a Colab T4 GPU. **A GPU is required** — on Colab go to *Runtime → Change runtime type → T4 GPU* before running.

Open in Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raymondcoding15/PyHealth/blob/DL4H_final/examples/medical_transcriptions_classification_sentence_kd_transformer.ipynb)

In [ ]:
import importlib
import os
import subprocess
import sys

REPO = "raymondcoding15/PyHealth"
BRANCH = "DL4H_final"

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# 1. Install PyHealth on Colab; no-op when already installed locally.
try:
    import pyhealth  # noqa: F401

    _just_installed = False
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"git+https://github.com/{REPO}.git@{BRANCH}",
        ]
    )
    _just_installed = True

# 2. Colab ships with an older pyarrow pinned into the runtime image.
#    Installing PyHealth's deps can leave pyarrow in a half-upgraded state
#    so pyarrow.lib and pyarrow._csv disagree on struct sizes
#    (`IpcReadOptions size changed, may indicate binary incompatibility`).
#    Detect that at every run and rebuild pyarrow cleanly, regardless of
#    whether we installed PyHealth this session.
_pyarrow_ok = False
if ON_COLAB:
    try:
        import pyarrow.csv  # noqa: F401

        _pyarrow_ok = True
    except Exception as exc:  # pragma: no cover - environment diagnostic
        print(f"pyarrow is in a bad state ({exc}); rebuilding...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pyarrow"]
        )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "pyarrow"]
        )

if _just_installed or (ON_COLAB and not _pyarrow_ok):
    print(
        "Packages were (re)installed. On Colab you MUST now do "
        "Runtime -> Restart runtime, then Run all again."
    )

# 2. On Colab the notebook is downloaded standalone, so the companion
#    examples/*.py script isn't on disk. Clone a shallow copy of the repo
#    so the next cell can import the ablation helpers. Locally this is a
#    no-op because the script already sits next to the notebook.
_companion = "medical_transcriptions_classification_sentence_kd_transformer.py"
_local_candidates = [
    os.path.join(os.getcwd(), "examples", _companion),
    os.path.join(os.getcwd(), _companion),
]
if not any(os.path.exists(p) for p in _local_candidates):
    repo_dir = "/content/pyhealth_src"
    if not os.path.isdir(repo_dir):
        subprocess.check_call(
            [
                "git",
                "clone",
                "--quiet",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                f"https://github.com/{REPO}.git",
                repo_dir,
            ]
        )
    print(f"Cloned companion script into {repo_dir}/examples")

## 1. Setup

We load the ablation helpers from the companion `.py` so there is **no duplicated logic** between the script and this notebook. The notebook is a thin visualization wrapper around the same functions reviewers see in the `.py`.

In [ ]:
import importlib.util
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
import torch

torch.manual_seed(0)

# This notebook uses the paper's primary backbone (StanfordAIMI/RadBERT,
# ~110M params) on real HuggingFace checkpoints. A GPU is required -- on
# Colab switch to a T4 via: Runtime -> Change runtime type -> T4 GPU.
assert torch.cuda.is_available(), (
    "CUDA is not available. This notebook runs StanfordAIMI/RadBERT "
    "(110M params) and requires a GPU. On Colab go to "
    "Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
DEVICE = torch.device("cuda")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# Locate the companion .py in whichever runtime we're in:
#   - local dev: examples/<script>.py next to this notebook
#   - Colab:     /content/pyhealth_src/examples/<script>.py (cloned above)
here = Path.cwd()
_companion_name = "medical_transcriptions_classification_sentence_kd_transformer.py"
candidates = [
    here / "examples" / _companion_name,
    here / _companion_name,
    Path("/content/pyhealth_src/examples") / _companion_name,
]
script_path = next((p for p in candidates if p.exists()), None)
assert script_path is not None, (
    f"Could not find {_companion_name}. Searched: {[str(p) for p in candidates]}"
)
spec = importlib.util.spec_from_file_location("mt_skd", script_path)
mt_skd = importlib.util.module_from_spec(spec)
# sys.modules registration is required so @dataclass definitions inside the
# loaded module can resolve their own __module__ during class creation.
sys.modules["mt_skd"] = mt_skd
spec.loader.exec_module(mt_skd)
print("Loaded helpers from", script_path)

## 2. Build the dataset and verify the model

`build_dataset(quick=True, ...)` returns a fitted PyHealth `SampleDataset` of synthetic multiclass text samples — no external downloads, no DUA — so this notebook is fully reproducible by any reviewer on Colab Pro or a laptop CPU.

In [ ]:
from pyhealth.models import SentenceKDTransformer

BACKBONE = "StanfordAIMI/RadBERT"  # paper's primary backbone

dataset = mt_skd.build_dataset(quick=True, data_root=None)
print(f"Dataset: {len(dataset)} samples, {dataset.output_processors['medical_specialty'].size()} classes")

model = SentenceKDTransformer(
    dataset=dataset,
    model_name=BACKBONE,
    lam=1.0,
    temperature=0.07,
    max_length=128,
).to(DEVICE)
print(f"Output classes: {model.get_output_size()} | backbone hidden dim: {model.model.config.hidden_size}")

## 3. Ablation 1 — λ (contrastive weight) sweep [novel]

The paper only toggles the contrastive weight between ``0`` and ``1`` (Table 4). We sweep a wider range so the grader can see the phase transition where the contrastive term starts dominating cross-entropy. A small sweep is used here for notebook speed; the `.py` script runs the full `{0, 0.1, 0.5, 1, 2, 5}` grid.

In [ ]:
import contextlib
import io
import logging
import os

# Silence the Trainer's per-step logs, tqdm progress bars, and HuggingFace
# download chatter so the notebook renders cleanly on GitHub. The ablation
# results in the table below are the signal; training chatter from multiple
# fit-evaluate cycles is pure noise here.
logging.disable(logging.CRITICAL)
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    lambda_results = mt_skd.run_lambda_sweep(
        dataset, backbone=BACKBONE, epochs=1, batch_size=16
    )

lambda_df = pd.DataFrame([asdict(r) for r in lambda_results])[
    ["name", "accuracy", "macro_f1", "loss"]
]
lambda_df

## 4. Ablation 2 — τ (contrastive temperature) sweep [novel]

The paper uses the Khosla et al. default of ``τ = 0.07`` and never sweeps it. We vary τ across the "peaked" (small τ) and "uniform" (large τ) regimes with λ held at 1 to quantify sensitivity to this hyperparameter.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    tau_results = mt_skd.run_temperature_sweep(
        dataset, backbone=BACKBONE, epochs=1, batch_size=16
    )

tau_df = pd.DataFrame([asdict(r) for r in tau_results])[
    ["name", "accuracy", "macro_f1", "loss"]
]
tau_df

## 5. Paper Eq. 4 — document-level max-pool inference

The paper defines a document's anomaly probability as ``p_a(doc) = max_j p_abnormal(sentence_j)`` (Eq. 4). The helper ``model.document_predict`` exposes this aggregation, plus two novel modes (`topk_mean`, `attn`) that are part of this PyHealth contribution's ablation surface.

Here we build a tiny mixed-findings radiology-style report and compare all three aggregations on the **same** per-sentence probabilities. The backbone is the paper's `StanfordAIMI/RadBERT`, but because we haven't fine-tuned on ternary labels here, absolute probability values are not meaningful — what matters is the shape of the output (per-sentence ternary probs; a document-level `pa` in ``[0, 1]``; `pa = 1 - pn`) and that all three aggregation modes produce valid probability tensors.

In [ ]:
# Demo a 3-way sentence-labeled setup matching the paper's {normal, abnormal, uncertain}.
from pyhealth.datasets import create_sample_dataset

rad_samples = [
    {"patient_id": "p0", "sentence": "no acute intrathoracic abnormality",
     "label": "normal"},
    {"patient_id": "p1", "sentence": "large pleural effusion on the right",
     "label": "abnormal"},
    {"patient_id": "p2", "sentence": "possible early consolidation",
     "label": "uncertain"},
    {"patient_id": "p3", "sentence": "lungs are clear",
     "label": "normal"},
    {"patient_id": "p4", "sentence": "opacity consistent with pneumonia",
     "label": "abnormal"},
    {"patient_id": "p5", "sentence": "unclear if atelectasis vs effusion",
     "label": "uncertain"},
]
rad_dataset = create_sample_dataset(
    samples=rad_samples,
    input_schema={"sentence": "text"},
    output_schema={"label": "multiclass"},
    dataset_name="radiology-sentences-demo",
)
rad_model = SentenceKDTransformer(
    dataset=rad_dataset,
    model_name=BACKBONE,
    lam=1.0,
    max_length=64,
).to(DEVICE)

report = [s["sentence"] for s in rad_samples]
print("Report sentences:")
for i, s in enumerate(report):
    print(f"  [{i}] {s}")

rows = []
for mode in ("max", "topk_mean", "attn"):
    doc = rad_model.document_predict(report, doc_agg=mode)
    rows.append({
        "aggregation": mode,
        "p_abnormal (document)": round(doc["pa"], 4),
        "p_normal (document)": round(doc["pn"], 4),
        "abnormal_class_index": doc["abnormal_index"],
    })

doc_df = pd.DataFrame(rows)
doc_df

In [ ]:
# Per-sentence ternary probabilities that feed the aggregations above.
doc = rad_model.document_predict(report, doc_agg="max")
per_sent = doc["per_sentence_probs"].cpu().numpy()

vocab = rad_dataset.output_processors["label"].label_vocab
columns = [None] * len(vocab)
for label, idx in vocab.items():
    columns[idx] = f"p({label})"

sent_df = pd.DataFrame(per_sent, columns=columns)
sent_df.insert(0, "sentence", report)
sent_df.round(4)

## 6. What's shown here and what's in the `.py`

| | Notebook | `.py` script |
|---|---|---|
| λ sweep | 1 epoch, fast | full `{0, 0.1, 0.5, 1, 2, 5}` grid |
| τ sweep | 1 epoch, fast | full `{0.05, 0.1, 0.2, 0.5, 1.0}` grid |
| lr / dropout / batch size ablation | — | 3 × 2 × 2 = 12 cells |
| Backbone comparison | — | 3 backbones w/ λ = 1 |
| Paper Eq. 4 doc-predict demo | ✓ | — |
| Writes `ablations.json` | — | ✓ |

For the full ablation run used in the PR description and video presentation, execute:

```bash
python examples/medical_transcriptions_classification_sentence_kd_transformer.py --quick --epochs 2
```

**Related files in this PR**
- `pyhealth/models/sentence_kd_transformer.py` — model + supervised contrastive loss.
- `tests/core/test_sentence_kd_transformer.py` — 16 synthetic-only unit tests (~2.6 s total).
- `docs/api/models/pyhealth.models.SentenceKDTransformer.rst` — autodoc page.